# Data Engineering Lab: High-Volume Ingestion & Analytics

##### Ingestión: Procesamiento de un dataset de 50M de registros (3GB RAW).

##### Optimización: Conversión de CSV a Apache Parquet (compresión Snappy) con motor PyArrow, logrando una reducción de tamaño del 50%.

##### Arquitectura Cloud: Implementación de un Data Lake en AWS S3 con particionamiento tipo Hive (comuna=X).

##### Seguridad: Gestión de credenciales mediante variables de entorno (.env) y acceso programático con Boto3.

##### Analytics: Ejecución de consultas SQL directas sobre S3 usando el motor DuckDB (httpfs), optimizando el costo por lectura (GET requests) y transferencia de datos.

## Crear el CSV de prueba

In [ ]:
import pandas as pd
import numpy as np
import time

n_rows = 50_000_000  # 10 millones de filas
print(f"Generando {n_rows} registros... Ten paciencia con tu RAM.")

data = {
    'id_vehiculo': np.random.randint(1, 1000, n_rows),
    'comuna': np.random.choice(['Concepcion', 'Talca', 'Santiago', 'Temuco', 'Antofagasta', 'Valparaiso'], n_rows),
    'velocidad': np.random.uniform(0, 120, n_rows).astype(np.float32),
    'lat': np.random.uniform(-37, -33, n_rows).astype(np.float32),
    'lon': np.random.uniform(-73, -70, n_rows).astype(np.float32),
    'timestamp': pd.date_range(start='2024-01-01', periods=n_rows, freq='s')
}

df = pd.DataFrame(data)
# Guardamos el CSV para comparar (¡Cuidado, esto va a pesar mucho!)
start = time.time()
df.to_csv('bioruta_mega_raw.csv', index=False)
print(f"✅ CSV creado en {time.time() - start:.2f} segundos.")

Generando 50000000 registros... Ten paciencia con tu RAM.
✅ CSV creado en 131.43 segundos.


0.27939677238464355

## El Experimento: CSV vs Parquet

In [8]:
print("Cargando 50 millones de filas... esto va a tardar.")
start = time.time()
# Si tienes poca RAM, podrías necesitar leerlo por trozos, 
# pero intentemos la carga completa primero:
df = pd.read_csv('bioruta_mega_raw.csv') 

print(f"Transformando a Parquet particionado...")
# El particionamiento aquí es vital. 
# Creará archivos manejables dentro de cada carpeta de comuna.
df.to_parquet('bioruta_50M_parquet', partition_cols=['comuna'], engine='fastparquet')
print(f"✅ ¡Hecho! Tiempo total: {time.time() - start:.2f}s")

Cargando 50 millones de filas... esto va a tardar.
Transformando a Parquet particionado...
✅ ¡Hecho! Tiempo total: 61.97s


Tardamos 62 segundos en procesar 50 millones de filas, es la velocidad que nos permite Parquet procesar los datos, mencionar que hubieron errores tambien, hay que tener instalado pyarrow en el contenedor sino tendremos un ImportError, este nos permite aprovechar la RAM para leer de manera rapida en disco, pero luego de instalarlo tuve el error ArrowKeyFunction que fue el mas interesante, ocurre por un choque de traduccion, Pandas usa tipos de datos de Python/Numpy, PyArrow es demasiado estricto, esto debido a que debe funcionar igual en Java, C++, Go, etc, por lo que es un estandar, ocurrio debido a que alguna columna tenia un tipo de dato que PyArrow no sabia como convertir a estandar universal, generalmente suele pasar con fechas o zona horaria, mezclando o tipos de numpy muy especificos.


Funciono con fastparquet pero tiene sus diferencias al procesar a Parquet con ese motor, sobretodo por que difiere en los metadatos, posiblemente pueda dar problemas con Spark.

## Normalizacion para que la transformacion del CSV -> Parquet sea mediante el motor 'pyarrow' que es el estandar actual

Spark usa internamente Apache Arrow para mover datos entre la JVM (Java) y Python (PySpark), al usar pyarrow garantizamos que no hayan discrepancias en los esquemas, timestamp mismatch y tambien eficiencia en la lectura de un cluster, ya que tiene soporte nativo de C++

In [9]:
import pyarrow.parquet as pq
import time
import pyarrow as pa
# 1. Cargar el DataFrame (si ya lo tienes en RAM, sáltate esto)
# df = pd.read_csv('bioruta_mega_raw.csv')

print("Normalizando tipos para PyArrow...")
# Convertimos a tipos estándar que Arrow ama
df['id_vehiculo'] = df['id_vehiculo'].astype('int32')
df['comuna'] = df['comuna'].astype(str)
df['velocidad'] = df['velocidad'].astype('float32')
df['lat'] = df['lat'].astype('float32')
df['lon'] = df['lon'].astype('float32')
# Las fechas son las que más suelen dar problemas de 'Extension Type'
df['timestamp'] = pd.to_datetime(df['timestamp'])

print("Exportando con motor PyArrow...")
start = time.time()

# Usamos la librería pyarrow directamente (es más estable para volúmenes grandes)
table = pa.Table.from_pandas(df, preserve_index=False)

pq.write_to_dataset(
    table, 
    root_path='bioruta_50M_pyarrow', 
    partition_cols=['comuna'],
    compression='snappy' # Snappy es el estándar de Spark
)

print(f"✅ ¡Éxito! PyArrow terminó en {time.time() - start:.2f}s")

Normalizando tipos para PyArrow...
Exportando con motor PyArrow...
✅ ¡Éxito! PyArrow terminó en 6.88s


Pasamos del csv de 3 gb a 1,3 aproximadamente, en un proceso mucho mas rapido debido a la normalizacion, no se redujo tanto el archivo porque los datos son aleatorios, Parquet no puede aprovechar el Dictionary Encoding o tomar un valor base e ir solo almacenando la suma para que de el valor siguiente, que optimizaria mucho, ademas los floats son dificiles de comprimir, pero en la vida real esto deberia funcionar mucho mejor reduciendo del (70-90)%.

## El Salto a la Nube (Subida Real)

obtendremos nuestras credenciales desde AWS, creando una persona en aws IAM, para ocupar de forma 'Local code' y generamos la Access Key ID publica y la Secret Access Key que es privada, como buena practica importaremos la libreria awscli de python para que las variables que son secretas no esten directamente en el codigo, esto creara un archivo oculto en ~/.aws/credentials que Boto3 leera por defecto.

### Verificamos conexion en S3

Listamos los buckets, debemos asegurarnos de dar permisos al usuario para que pueda listar la base de datos antes, la manera mas profesional es creando esa regla.

In [13]:
import boto3

# No necesitas pasar parámetros aquí si las variables están bien configuradas
s3 = boto3.resource('s3')

try:
    # Listar tus buckets para probar conexión
    for bucket in s3.buckets.all():
        print(f"Conexión exitosa. Bucket encontrado: {bucket.name}")
except Exception as e:
    print(f"Error de conexión: {e}")

Conexión exitosa. Bucket encontrado: wesrugby-media


In [14]:
import os

# Configuración
REGION = 'us-east-1'  
BUCKET_NAME = 'bioruta-data-lake-50m-res' # debe ser unico en todo el mundo
LOCAL_FOLDER = 'bioruta_50M_pyarrow'

s3 = boto3.client('s3', region_name=REGION)

# 1. Crear el Bucket
try:
    print(f"Creando bucket: {BUCKET_NAME}...")
    if REGION == 'us-east-1':
        s3.create_bucket(Bucket=BUCKET_NAME)
    else:
        location = {'LocationConstraint': REGION}
        s3.create_bucket(Bucket=BUCKET_NAME, CreateBucketConfiguration=location)
    print("✅ Bucket creado con éxito.")
except s3.exceptions.BucketAlreadyExists:
    print("❌ El nombre ya existe a nivel global. Elige otro nombre más específico.")
except Exception as e:
    print(f"❌ Error: {e}")


Creando bucket: bioruta-data-lake-50m-res...
✅ Bucket creado con éxito.


## Subida masiva de archivos a S3
S3 funciona con un espacio infinito, puede pesar 1 byte o 100 petabytes, solo que se factura mensual la cantidad que este almacenada, se paga aprox 0.023 usd por GB, por lo que lo que subire costara 0.034 USD al mes equivalente a 1.5 GB (50 millones de registros en Parquet)

In [15]:
# 2. Subida Masiva (con la estructura de particiones)
print(f"\nIniciando subida de 50 millones de registros...")

for root, dirs, files in os.walk(LOCAL_FOLDER):
    for file in files:
        full_path = os.path.join(root, file)
        
        # Crear la ruta de S3 respetando las carpetas (comuna=X/archivo.parquet)
        s3_key = os.path.relpath(full_path, LOCAL_FOLDER).replace("\\", "/")
        
        print(f"Subiendo: {s3_key}")
        s3.upload_file(full_path, BUCKET_NAME, s3_key)

print("\n🚀 ¡TODO LISTO! Tus 50 millones de filas están en el Data Lake.")


Iniciando subida de 50 millones de registros...
Subiendo: comuna=Antofagasta/b2f00b34dcee4f6592b59dd59fd16852-0.parquet
Subiendo: comuna=Concepcion/b2f00b34dcee4f6592b59dd59fd16852-0.parquet
Subiendo: comuna=Santiago/b2f00b34dcee4f6592b59dd59fd16852-0.parquet
Subiendo: comuna=Talca/b2f00b34dcee4f6592b59dd59fd16852-0.parquet
Subiendo: comuna=Temuco/b2f00b34dcee4f6592b59dd59fd16852-0.parquet
Subiendo: comuna=Valparaiso/b2f00b34dcee4f6592b59dd59fd16852-0.parquet

🚀 ¡TODO LISTO! Tus 50 millones de filas están en el Data Lake.


Importante saber que AWS cobra ademas por peticiones PUT, GET, LIST, etc. asi que ten cuidado con las pruebas y no olvides eliminar el bucket si ya no lo necesitas, la ventaja de Parquet aca cobra sentido porque al por ejemplo particionar por comunas, el sistema no tiene que hacer un GET a los archivos de las otras columnas y se podria optimizar mucho mas todavia con los metadatos de parquet que estan indexados

## Script para auditar el Data Lake

Listamos primero por resource que tiene un enfoque orientado a objetos y es intuitivo para entender, pero su rendimiento no es el mejor para Big Data, apenas recibe nuevas funciones y es facil de entender (pythonic)

In [19]:
s3 = boto3.resource('s3')
bucket = s3.Bucket(BUCKET_NAME)

for obj in bucket.objects.all():
    print(obj.key, obj.size/1024/1024, "MB")

comuna=Antofagasta/b2f00b34dcee4f6592b59dd59fd16852-0.parquet 212.77892208099365 MB
comuna=Concepcion/b2f00b34dcee4f6592b59dd59fd16852-0.parquet 212.88249397277832 MB
comuna=Santiago/b2f00b34dcee4f6592b59dd59fd16852-0.parquet 212.85814380645752 MB
comuna=Talca/b2f00b34dcee4f6592b59dd59fd16852-0.parquet 212.80388736724854 MB
comuna=Temuco/b2f00b34dcee4f6592b59dd59fd16852-0.parquet 212.79279041290283 MB
comuna=Valparaiso/b2f00b34dcee4f6592b59dd59fd16852-0.parquet 212.74924564361572 MB


Repetimos el proceso pero con client que esta optimizado para Big Data, siempre al dia con AWS y orientado a servicios API, pero mas dificil de entender

In [20]:

# Configuración
BUCKET_NAME = 'bioruta-data-lake-50m-res' 
s3 = boto3.client('s3')

print(f"--- Inventario del Data Lake: {BUCKET_NAME} ---")

# Paginador para manejar buckets con muchos archivos (buena práctica)
paginator = s3.get_paginator('list_objects_v2')
pages = paginator.paginate(Bucket=BUCKET_NAME)

total_size_bytes = 0
total_files = 0
comunas_encontradas = set()

for page in pages:
    if 'Contents' in page:
        for obj in page['Contents']:
            total_files += 1
            total_size_bytes += obj['Size']
            
            # Extraer el nombre de la comuna de la ruta (prefijo)
            # Ejemplo de ruta: comuna=Santiago/part-0.parquet
            if 'comuna=' in obj['Key']:
                nombre_comuna = obj['Key'].split('/')[0].replace('comuna=', '')
                comunas_encontradas.add(nombre_comuna)

# Resultados
size_gb = total_size_bytes / (1024**3)
print(f"✅ Total de archivos: {total_files}")
print(f"✅ Tamaño total en S3: {size_gb:.4f} GB")
print(f"✅ Particiones detectadas: {', '.join(comunas_encontradas)}")
print("------------------------------------------")

--- Inventario del Data Lake: bioruta-data-lake-50m-res ---
✅ Total de archivos: 6
✅ Tamaño total en S3: 1.2469 GB
✅ Particiones detectadas: Valparaiso, Santiago, Concepcion, Antofagasta, Temuco, Talca
------------------------------------------


## Ejemplo de consulta en la nube con duckdb

In [22]:
import os
import duckdb
from dotenv import load_dotenv

# Cargar variables desde el archivo .env
load_dotenv()

# Recuperar variables
key_id = os.getenv('AWS_ACCESS_KEY_ID')
secret = os.getenv('AWS_SECRET_ACCESS_KEY')
region = os.getenv('AWS_REGION')
bucket = os.getenv('BUCKET_NAME')

# Iniciar DuckDB
con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# Configurar el secreto sin "hardcodear" nada
con.execute(f"""
    CREATE OR REPLACE SECRET (
        TYPE S3,
        KEY_ID '{key_id}',
        SECRET '{secret}',
        REGION '{region}'
    );
""")

# Consulta SQL con Hive Partitioning
query = f"""
    SELECT 
        comuna, 
        COUNT(*) as total_filas, 
        ROUND(AVG(velocidad), 2) as avg_speed
    FROM read_parquet('s3://{bucket}/**/*.parquet', hive_partitioning=1)
    GROUP BY comuna
    ORDER BY avg_speed DESC;
"""

print("🚀 Consultando el Data Lake de forma segura...")
df_result = con.execute(query).df()
print("\n--- RESULTADOS FINALES ---")
print(df_result)

🚀 Consultando el Data Lake de forma segura...

--- RESULTADOS FINALES ---
        comuna  total_filas  avg_speed
0   Concepcion      8336324      60.01
1  Antofagasta      8331990      60.01
2       Temuco      8332522      60.01
3   Valparaiso      8330786      60.01
4     Santiago      8335319      60.00
5        Talca      8333059      59.98


El tiempo de la consulta en este experimento fue de 3 minutos 15 segundos, pero debemos tener en cuenta que se estaba comunicando con un servidor en la nube en estados unidos, metadatos de 50 millones de registros, descargo solo las columnas necesarias, velocidad y comuna y calculo los promedios sobre la marcha.